In [11]:
import torch
import math
from torch.distributions import Normal
import torch.nn as nn

In [3]:
def brownian_motion(n_paths, n_steps, seed, S0=100.0, sigma=0.2, T=30/252):
    dt = T / n_steps
    g = torch.Generator().manual_seed(seed)
    Z = torch.randn(n_paths, n_steps, generator=g)
    log_returns = -0.5 * sigma**2 * dt + sigma * dt**0.5 * Z
    log_paths = torch.cumsum(log_returns, dim=1)
    log_paths = torch.cat([torch.zeros(n_paths, 1), log_paths], dim=1)
    return S0 * torch.exp(log_paths)

In [4]:
paths = brownian_motion(100_000, 30, seed=0)
print(paths.shape)                          
print(paths[:, 0].unique())                  
print(paths[:, -1].mean())                  
print(torch.log(paths[:, -1] / 100).std()) 

torch.Size([100000, 31])
tensor([100.])
tensor(99.9671)
tensor(0.0690)


In [5]:
def bs_call_price(S0, K, T, sigma):
    d1 = (math.log(S0 / K) + 0.5 * sigma**2 * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))  
    return S0 * N(d1) - K * N(d2)

def bs_delta(paths, K, T, sigma):
    n_steps = paths.shape[1] - 1
    dt = T / n_steps
    tau = T - torch.arange(n_steps) * dt     
    S = paths[:, :-1]                        
    d1 = (torch.log(S / K) + 0.5 * sigma**2 * tau) / (sigma * torch.sqrt(tau))
    return Normal(0.0, 1.0).cdf(d1)

In [6]:
K = 100.0
T = 30/252
sigma = 0.2

premium = bs_call_price(100.0, K, T, sigma)
print(premium)                        

paths = brownian_motion(100_000, 30, seed=0)
delta = bs_delta(paths, K, T, sigma)
print(delta.shape)                    
print(delta[:, 0].unique())           
print(delta.min(), delta.max())        
print(delta[paths[:, -1] > 110, -1].mean())   
print(delta[paths[:, -1] < 90, -1].mean()) 

2.7524171533585005
torch.Size([100000, 30])
tensor([0.5138])
tensor(0.) tensor(1.)
tensor(1.)
tensor(3.9357e-11)


In [8]:
def hedging_pnl(paths, delta, c, K, premium):
   
    S = paths[:, :-1]                           
    dS = paths[:, 1:] - paths[:, :-1]         

    gains = (delta * dS).sum(dim=1)            

    prev = torch.cat([torch.zeros_like(delta[:, :1]), delta[:, :-1]], dim=1)
    trades = (delta - prev).abs()               
    costs = c * (trades * S).sum(dim=1)        

    payoff = torch.clamp(paths[:, -1] - K, min=0)   

    pnl = premium - payoff + gains - costs
    turnover = trades.sum(dim=1)
    return pnl, turnover

In [9]:
for c in [0.0, 0.001, 0.005]:
    pnl, turnover = hedging_pnl(paths, delta, c, K, premium)
    print(f"c={c}: mean PnL {pnl.mean():.4f}, std {pnl.std():.4f}, turnover {turnover.mean():.3f}")

c=0.0: mean PnL 0.0004, std 0.4307, turnover 2.224
c=0.001: mean PnL -0.2219, std 0.4450, turnover 2.224
c=0.005: mean PnL -1.1110, std 0.5677, turnover 2.224


In [10]:
def cvar(pnl, alpha=0.95):
    k = int((1 - alpha) * len(pnl))
    return torch.topk(-pnl, k).values.mean()

val_paths  = brownian_motion(50_000, 30, seed=2)
test_paths = brownian_motion(50_000, 30, seed=123)

def evaluate(paths, delta, c):
    pnl, turn = hedging_pnl(paths, delta, c, K, premium)
    return f"mean {pnl.mean():.4f}, std {pnl.std():.4f}, CVaR95 {cvar(pnl):.4f}, turnover {turn.mean():.3f}"

bs_test = bs_delta(test_paths, K, T, sigma)
for c in [0.0, 0.001, 0.005]:
    print(f"BS c={c}:", evaluate(test_paths, bs_test, c))

BS c=0.0: mean -0.0033, std 0.4314, CVaR95 1.0072, turnover 2.223
BS c=0.001: mean -0.2255, std 0.4462, CVaR95 1.3126, turnover 2.223
BS c=0.005: mean -1.1144, std 0.5701, CVaR95 2.5877, turnover 2.223


In [12]:
class HedgeMLP(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1), nn.Sigmoid())

    def forward(self, paths, K, T):
        n_paths, n_steps = paths.shape[0], paths.shape[1] - 1
        dt = T / n_steps
        prev = torch.zeros(n_paths)
        deltas = []
        for t in range(n_steps):
            tau = torch.full((n_paths,), (T - t * dt) / T)        
            logm = 10 * torch.log(paths[:, t] / K)              
            x = torch.stack([logm, tau, prev], dim=1)
            prev = self.net(x).squeeze(1)
            deltas.append(prev)
        return torch.stack(deltas, dim=1)                        

In [16]:
def train(model, c, loss_type="mse", iters=3000, batch=10_000, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=iters)
    for i in range(iters):
        p = brownian_motion(batch, 30, seed=10_000 + i)
        pnl, _ = hedging_pnl(p, model(p, K, T), c, K, premium)
        loss = (pnl ** 2).mean() if loss_type == "mse" else cvar(pnl)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if i % 500 == 0:
            print(i, loss.item())
    return model

In [17]:
model = train(HedgeMLP(), c=0.0)
with torch.no_grad():
    print("MLP c=0:", evaluate(val_paths, model(val_paths, K, T), 0.0))
    print("BS  c=0:", evaluate(val_paths, bs_delta(val_paths, K, T, sigma), 0.0))

0 4.391081809997559
500 0.1955316811800003
1000 0.18887943029403687
1500 0.18804140388965607
2000 0.18934430181980133
2500 0.18261827528476715
MLP c=0: mean 0.0017, std 0.4327, CVaR95 1.0010, turnover 2.219
BS  c=0: mean 0.0015, std 0.4313, CVaR95 0.9982, turnover 2.221


In [18]:
model_005 = train(HedgeMLP(), c=0.005)
with torch.no_grad():
    print("MLP c=0.005:", evaluate(val_paths, model_005(val_paths, K, T), 0.005))
    print("BS  c=0.005:", evaluate(val_paths, bs_delta(val_paths, K, T, sigma), 0.005))

0 4.366687297821045
500 0.8451789021492004
1000 0.8186128735542297
1500 0.8290509581565857
2000 0.8306023478507996
2500 0.820503830909729
MLP c=0.005: mean -0.6466, std 0.6282, CVaR95 2.0519, turnover 1.296
BS  c=0.005: mean -1.1084, std 0.5673, CVaR95 2.5580, turnover 2.221


In [20]:
def train(model, c, loss_type="mse", iters=3000, batch=10_000, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=iters)
    for i in range(iters):
        p = brownian_motion(batch, 30, seed=10_000 + i)
        pnl, _ = hedging_pnl(p, model(p, K, T), c, K, premium)
        if loss_type == "mse":
            loss = (pnl ** 2).mean()
        elif loss_type == "var":
            loss = pnl.var()
        else:
            loss = cvar(pnl)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if i % 500 == 0:
            print(i, loss.item())
    return model

In [21]:
m_var = train(HedgeMLP(), c=0.005, loss_type="var")
print("var :", evaluate(val_paths, m_var(val_paths, K, T), 0.005))

m_cvar = train(HedgeMLP(), c=0.005, loss_type="cvar", batch=20_000)
print("cvar:", evaluate(val_paths, m_cvar(val_paths, K, T), 0.005))

0 4.3568010330200195
500 0.2553914189338684
1000 0.23756137490272522
1500 0.23361563682556152
2000 0.23305317759513855
2500 0.22720138728618622
var : mean -0.9952, std 0.4795, CVaR95 2.0899, turnover 1.992
0 5.766733169555664
500 1.864111065864563
1000 1.8560576438903809
1500 1.856868028640747
2000 1.8549143075942993
2500 1.8521757125854492
cvar: mean -0.7895, std 0.5874, CVaR95 1.8523, turnover 1.581
